ban đầu cho AI nó ăn toàn bộ  list ingredient_id, name id
tiếp tho ta chạy để lấy toàn bộ id trong bảng dish
lưu json
lấy dữ liệu json tham chiếu , select trong db dish in ra các nguyên liệu dưới dạng
dish_id , name , ingredients[],
đưa toàn bộ  vào 1 list file txt dạng sau
promt : câu promt này yêu AI đọc dữ liệu đã được cho ăn về tất cả cac ingredient yêu cầu chỉ dựa  các ingredient đó
 duyệt qua toàn bộ các ingredient trong dish này xem có cái nào tương ứng ko  nếu có thì thay thay thế thành id của ingedient đó 
 kết quả là dạng dữ liệu ingrdient tương ứng trong chính data dc gửi, 
 

In [1]:
import sqlite3
import os

# ── Cấu hình ──────────────────────────────────────────────
DB_PATH     = "../cookpad_clean.db"        # 👈 Thay bằng đường dẫn DB của bạn
OUTPUT_PATH = "ingredients_export.txt"
# ──────────────────────────────────────────────────────────

def export_ingredients(db_path: str, output_path: str):
    if not os.path.exists(db_path):
        print(f"Lỗi: Không tìm thấy file database '{db_path}'")
        return

    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    cursor.execute("SELECT id, name, name_en FROM ingredients ORDER BY id")
    rows = cursor.fetchall()
    conn.close()

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(f"{'ID':<6} {'NAME':<40} {'NAME_EN'}\n")
        f.write("-" * 80 + "\n")
        for row in rows:
            id_, name, name_en = row
            name_en = name_en or ""
            f.write(f"{id_:<6} {name:<40} {name_en}\n")

    print(f"✅ Đã xuất {len(rows)} ingredients ra file: {output_path}")

export_ingredients(DB_PATH, OUTPUT_PATH)

✅ Đã xuất 1120 ingredients ra file: ingredients_export.txt


In [4]:
import sqlite3
import json
import os

# ── Cấu hình ──────────────────────────────────────────────
DB_PATH     = "../cookpad_clean.db"        # 👈 Thay bằng đường dẫn DB của bạn
OUTPUT_PATH = "dishes_export.json"
# ──────────────────────────────────────────────────────────

def export_dishes(db_path: str, output_path: str):
    if not os.path.exists(db_path):
        print(f"Lỗi: Không tìm thấy file database '{db_path}'")
        return

    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    cursor.execute("SELECT id, title FROM dishes ORDER BY id")
    rows = cursor.fetchall()
    conn.close()

    dishes = [{"id": row[0], "name": row[1]} for row in rows]

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(dishes, f, ensure_ascii=False, indent=2)

    print(f"✅ Đã xuất {len(dishes)} dishes ra file: {output_path}")

export_dishes(DB_PATH, OUTPUT_PATH)

✅ Đã xuất 8260 dishes ra file: dishes_export.json


In [3]:
"""
export_prompt_batches.py
=========================
Query DB lấy recipes + ingredients thô
→ Xuất TXT batches (30 món/file) với prompt yêu cầu AI map tên nguyên liệu → ingredient_id

Cấu trúc DB:
  recipes     (id TEXT PK, title TEXT, ...)
  ingredients (id INTEGER PK, recipe_id TEXT, amount TEXT, unit TEXT, name TEXT, raw TEXT)

Cấu trúc ingredient chuẩn bạn đã feed cho AI:
  id, name, name_en
"""

import sqlite3
import json
import math
from pathlib import Path

# ══════════════════════════════════════════════════════════════
# CẤU HÌNH
# ══════════════════════════════════════════════════════════════
DB_PATH         = "./cookpad.db"       # <-- sửa đường dẫn DB
PROMPT_DIR      = "./prompts_batches"  # thư mục lưu file TXT
FILTER_JSON     = "./dishes_export.json"      # <-- file JSON chứa danh sách id cần lấy
BATCH_SIZE      = 30                   # số món mỗi batch

# ══════════════════════════════════════════════════════════════
# PROMPT TEMPLATE
# ══════════════════════════════════════════════════════════════
PROMPT_TEMPLATE = """\
Dựa vào danh sách ingredient chuẩn tôi đã cung cấp (có id, name, name_en),
hãy duyệt qua từng nguyên liệu trong mỗi món bên dưới và map tên thô → ingredient_id tương ứng.

QUY TẮC:
- Nếu tìm thấy nguyên liệu khớp (kể cả khớp gần đúng) → trả về ingredient_id (integer)
- Nếu không tìm thấy → trả về null
- Chỉ trả về JSON array duy nhất, KHÔNG kèm văn bản thừa hay markdown

OUTPUT FORMAT:
[
  {{
    "recipe_id": "<string>",
    "dish_name": "<string>",
    "raw_ingredient_id": <int>,
    "raw_name": "<string>",
    "mapped_ingredient_id": <int hoặc null>
  }}
]

## BATCH {batch_num}/{total_batches} — Món {start}–{end} / {total_recipes} tổng
{payload_json}
"""

# ══════════════════════════════════════════════════════════════
# ĐỌC DANH SÁCH ID CẦN LỌC TỪ FILE JSON
# ══════════════════════════════════════════════════════════════
with open(FILTER_JSON, encoding="utf-8") as f:
    filter_data = json.load(f)

allowed_ids = {str(item["id"]) for item in filter_data}
print(f"Số món trong filter JSON : {len(allowed_ids)}")

# ══════════════════════════════════════════════════════════════
# QUERY DB
# ══════════════════════════════════════════════════════════════
print("Kết nối DB...")
con = sqlite3.connect(DB_PATH)
con.row_factory = sqlite3.Row

# Dùng placeholders để filter trực tiếp trong SQL
placeholders = ",".join("?" * len(allowed_ids))
rows = con.execute(f"""
    SELECT
        r.id     AS recipe_id,
        r.title  AS dish_name,
        i.id     AS raw_ingredient_id,
        i.name   AS ingredient_name,
        i.raw    AS ingredient_raw,
        i.amount,
        i.unit
    FROM recipes r
    JOIN ingredients i ON i.recipe_id = r.id
    WHERE r.id IN ({placeholders})
    ORDER BY r.title, i.id
""", list(allowed_ids)).fetchall()
con.close()

print(f"Tổng ingredient records (đã lọc): {len(rows)}")

# Nhóm theo recipe
recipes_map = {}
for row in rows:
    rid = row["recipe_id"]
    if rid not in recipes_map:
        recipes_map[rid] = {
            "recipe_id": rid,
            "dish_name": row["dish_name"],
            "ingredients": []
        }
    # Ưu tiên dùng raw nếu có, fallback về name
    raw_name = row["ingredient_raw"] or row["ingredient_name"]
    recipes_map[rid]["ingredients"].append({
        "raw_ingredient_id": row["raw_ingredient_id"],
        "raw_name":          raw_name,
    })

all_recipes   = list(recipes_map.values())
total_recipes = len(all_recipes)
total_batches = math.ceil(total_recipes / BATCH_SIZE)

print(f"Tổng món (recipes)    : {total_recipes}")
print(f"Batch size            : {BATCH_SIZE}")
print(f"Số file sẽ tạo        : {total_batches}")

# ══════════════════════════════════════════════════════════════
# XUẤT FILE TXT
# ══════════════════════════════════════════════════════════════
Path(PROMPT_DIR).mkdir(exist_ok=True)

for i in range(total_batches):
    start  = i * BATCH_SIZE
    end    = min(start + BATCH_SIZE, total_recipes)
    batch  = all_recipes[start:end]

    # Flatten: mỗi item là 1 ingredient của 1 recipe
    flat_items = []
    for recipe in batch:
        for ing in recipe["ingredients"]:
            flat_items.append({
                "recipe_id":          recipe["recipe_id"],
                "dish_name":          recipe["dish_name"],
                "raw_ingredient_id":  ing["raw_ingredient_id"],
                "raw_name":           ing["raw_name"],
            })

    prompt = PROMPT_TEMPLATE.format(
        batch_num     = i + 1,
        total_batches = total_batches,
        start         = start + 1,
        end           = end,
        total_recipes = total_recipes,
        payload_json  = json.dumps(flat_items, ensure_ascii=False, indent=2),
    )

    out_path = Path(PROMPT_DIR) / f"prompt_batch_{i+1:03d}.txt"
    out_path.write_text(prompt, encoding="utf-8")
    print(f"  ✅ {out_path.name}  ({len(batch)} món, {len(flat_items)} ingredients)")

print(f"\n📁 Prompts đã lưu tại: {Path(PROMPT_DIR).resolve()}")
print()
print("📌 Bước tiếp theo:")
print(f"  1. Mở từng file trong '{PROMPT_DIR}/'")
print(f"  2. Paste vào Claude (đã được feed danh sách ingredient chuẩn)")
print(f"  3. Lưu response JSON thành filled_batch_001.json, _002.json ...")

Số món trong filter JSON : 8282
Kết nối DB...
Tổng ingredient records (đã lọc): 60366
Tổng món (recipes)    : 8282
Batch size            : 30
Số file sẽ tạo        : 277
  ✅ prompt_batch_001.txt  (30 món, 291 ingredients)
  ✅ prompt_batch_002.txt  (30 món, 326 ingredients)
  ✅ prompt_batch_003.txt  (30 món, 303 ingredients)
  ✅ prompt_batch_004.txt  (30 món, 304 ingredients)
  ✅ prompt_batch_005.txt  (30 món, 387 ingredients)
  ✅ prompt_batch_006.txt  (30 món, 251 ingredients)
  ✅ prompt_batch_007.txt  (30 món, 221 ingredients)
  ✅ prompt_batch_008.txt  (30 món, 207 ingredients)
  ✅ prompt_batch_009.txt  (30 món, 279 ingredients)
  ✅ prompt_batch_010.txt  (30 món, 247 ingredients)
  ✅ prompt_batch_011.txt  (30 món, 282 ingredients)
  ✅ prompt_batch_012.txt  (30 món, 248 ingredients)
  ✅ prompt_batch_013.txt  (30 món, 272 ingredients)
  ✅ prompt_batch_014.txt  (30 món, 211 ingredients)
  ✅ prompt_batch_015.txt  (30 món, 351 ingredients)
  ✅ prompt_batch_016.txt  (30 món, 254 ingredients

In [4]:
import sqlite3
import re
import unicodedata

DB_PATH = "./cookpad.db"

SPICE_KEYWORDS = [
    "muối", "đường", "bột ngọt", "mì chính", "hạt nêm", "bột nêm",
    "nước mắm", "xì dầu",  "tương hoisin",
     "tương cà", "mù tạt", "mustard", "mayonnaise",
    
    
    "bột năng",  "bột chiên giòn",
    "bột nở", "men nở", 
    "whipping cream",
    "tiêu", "paprika", "nghệ bột", "hồi", "đinh hương",
    "thảo quả", "lá nguyệt quế", "oregano","rosemary", "thyme",
     "sake"
    
  , "viên súp", "nước dùng", "nước lèo","gia vị","Gia vị","Gia Vị"
]

OPTIONAL_PATTERNS = [
    r'\(?\s*tùy\s*chọn\s*\)?',
    r'\(?\s*tùy\s*thích\s*\)?',
    r'\(?\s*tùy\s*khẩu\s*vị\s*\)?',
    r'\(?\s*optional\s*\)?',
    r'\(?\s*nếu\s*thích\s*\)?',
    r'\(?\s*có\s*thể\s*bỏ\s*qua\s*\)?',
    r'\(?\s*hoặc\s*không\s*\)?',
]

# ─── VIETNAMESE CHARS ────────────────────────────────────────────────────────
VI_CHARS = "àáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđ"


def clean_special_chars(name: str) -> str:
    """Xóa số và ký tự đặc biệt, chỉ giữ chữ cái (kể cả tiếng Việt) và khoảng trắng."""
    # Xóa tất cả trừ chữ cái latin, chữ Việt có dấu, khoảng trắng
    name = re.sub(rf"[^a-zA-Z{VI_CHARS}\s]", "", name)
    # Collapse whitespace
    return re.sub(r"\s+", " ", name).strip()


def is_spice(name: str) -> bool:
    name_norm = unicodedata.normalize("NFC", name.strip().lower())
    name_clean = re.sub(r'\(.*?\)', '', name_norm).strip()
    for kw in SPICE_KEYWORDS:
        if re.search(rf'(^|[\s,])' + re.escape(kw) + r'($|[\s,])', name_clean):
            return True
    return False


def strip_optional_text(name: str) -> str:
    for pattern in OPTIONAL_PATTERNS:
        name = re.sub(pattern, '', name, flags=re.IGNORECASE)
    return re.sub(r'\s+', ' ', name).strip()


def clean_ingredients(db_path: str, dry_run: bool = True):
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    cur = conn.cursor()

    cur.execute("SELECT id, name FROM ingredients")
    all_rows = cur.fetchall()
    print(f"📦 Tổng ingredients: {len(all_rows):,}\n")

    to_delete = []
    to_update = []

    for row in all_rows:
        ing_id = row["id"]
        raw    = row["name"]

        if not raw:
            continue

        # Pipeline: optional → special chars → spice check
        cleaned = strip_optional_text(raw)
        cleaned = clean_special_chars(cleaned)      # ← bước mới

        if is_spice(cleaned):
            print(f"  🗑  GIA VỊ   [{ing_id}] '{raw}'")
            to_delete.append(ing_id)
            continue

        if cleaned != raw:
            print(f"  ✏️  CLEAN    [{ing_id}] '{raw}' → '{cleaned}'")
            to_update.append((cleaned, ing_id))

    print(f"\n{'='*50}")
    print(f"🗑  Gia vị cần xóa  : {len(to_delete):,}")
    print(f"✏️  Cần cập nhật    : {len(to_update):,}")

    if not dry_run:
        cur.execute("PRAGMA foreign_keys = OFF")

        if to_delete:
            placeholders = ",".join("?" * len(to_delete))
            cur.execute(f"DELETE FROM ingredients WHERE id IN ({placeholders})", to_delete)
            print(f"\n✅ Đã xóa {cur.rowcount} gia vị.")

        for new_name, ing_id in to_update:
            cur.execute("UPDATE ingredients SET name = ? WHERE id = ?", (new_name, ing_id))
        print(f"✅ Đã cập nhật {len(to_update)} dòng.")

        cur.execute("PRAGMA foreign_keys = ON")
        conn.commit()

        cur.execute("SELECT COUNT(*) FROM ingredients")
        print(f"📊 ingredients còn lại: {cur.fetchone()[0]:,}")
    else:
        print("\n⚠️  DRY-RUN — chưa thay đổi gì. Set dry_run=False để chạy thật.")

    conn.close()


if __name__ == "__main__":
    clean_ingredients(DB_PATH, dry_run=False)
    # clean_ingredients(DB_PATH, dry_run=False)

📦 Tổng ingredients: 48,333

  🗑  GIA VỊ   [216] 'Gói gia vị mì'
  🗑  GIA VỊ   [237] 'Gia vị ướp'
  🗑  GIA VỊ   [366] 'Gia vị thông thường'
  🗑  GIA VỊ   [395] 'Gia vị nấu sốt ướp'
  🗑  GIA VỊ   [422] 'Gia vị cơm'
  🗑  GIA VỊ   [425] 'Gia vị ướp cá'
  🗑  GIA VỊ   [509] 'gia vị ragu'
  🗑  GIA VỊ   [716] 'trai dưa tươi lây nươc gia vị'
  🗑  GIA VỊ   [926] 'Gia vị'
  🗑  GIA VỊ   [938] 'Gia vị'
  🗑  GIA VỊ   [942] 'Gia vị'
  🗑  GIA VỊ   [962] 'Gia vị'
  🗑  GIA VỊ   [968] 'Gia vị'
  🗑  GIA VỊ   [986] 'Gia vị Italian seasoning mixed'
  🗑  GIA VỊ   [1041] 'Gia vị ướp thịt'
  🗑  GIA VỊ   [1216] 'Gia vị'
  🗑  GIA VỊ   [1244] 'Gia vị'
  🗑  GIA VỊ   [1362] 'Gia vị'
  🗑  GIA VỊ   [1396] 'Gia vị muốitiêuhạt nêm'
  🗑  GIA VỊ   [1521] 'Gia vị thông thường'
  🗑  GIA VỊ   [1550] 'Gia vị'
  🗑  GIA VỊ   [1637] 'Gia vị'
  🗑  GIA VỊ   [1654] 'Gia vị'
  🗑  GIA VỊ   [1660] 'Gia vị'
  🗑  GIA VỊ   [1664] 'Gia vị'
  🗑  GIA VỊ   [1759] 'Gia vị'
  🗑  GIA VỊ   [1815] 'Gia vị'
  🗑  GIA VỊ   [1822] 'Gia vị ớt sả hành

OperationalError: database is locked

In [6]:
import sqlite3
import json

DB_PATH = "./cookpad.db"
JSON_PATH = "./dishes_export.json"

# đọc id hợp lệ từ json
with open(JSON_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

valid_ids = {item["id"] for item in data if item.get("id")}

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

# tạo placeholder cho SQL
placeholders = ",".join(["?"] * len(valid_ids))

# 1. xóa ingredient không có recipe_id hợp lệ
cur.execute(f"""
DELETE FROM ingredients
WHERE recipe_id NOT IN ({placeholders})
""", tuple(valid_ids))

print("Deleted ingredients:", cur.rowcount)

# 2. xóa recipes không nằm trong json
cur.execute(f"""
DELETE FROM recipes
WHERE id NOT IN ({placeholders})
""", tuple(valid_ids))

print("Deleted recipes:", cur.rowcount)

conn.commit()
conn.close()

Deleted ingredients: 8790
Deleted recipes: 1730


In [1]:
# create json file containe dish-ingredient
import sqlite3
import json

# Kết nối tới file SQLite của bạn
conn = sqlite3.connect("../cookpad_clean.db")  # ← đổi tên file .db cho đúng
conn.row_factory = sqlite3.Row  # để lấy dữ liệu dạng dict

cursor = conn.cursor()

# Query toàn bộ bảng
cursor.execute("SELECT * FROM dish_ingredient")
rows = cursor.fetchall()

# Chuyển sang list of dict
data = [dict(row) for row in rows]

# Ghi ra file JSON
output_path = "dish_ingredient.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

conn.close()
print(f"✅ Đã xuất {len(data)} dòng ra file: {output_path}")

✅ Đã xuất 32177 dòng ra file: dish_ingredient.json


In [4]:
import sqlite3
import json
import os
from collections import defaultdict

DB_PATH = './cookpad.db'
DISH_INGREDIENT_JSON = './dish_ingredient.json'
OUTPUT_DIR = 'ai_prompts'
BATCH_SIZE = 30

prompt_template = """
Bạn là một chuyên gia về dữ liệu ẩm thực Việt Nam. 
Tôi có một danh sách các nguyên liệu thô (raw ingredients) từ bảng 'ingredients'. 
Nhiệm vụ của bạn: Ánh xạ 'name' của nguyên liệu thô này sang 'standard_ingredient_id' và 'standard_name' từ danh mục 1500 nguyên liệu chuẩn mà bạn đã được học.

DỮ LIỆU ĐẦU VÀO (JSON):
{input_data}

YÊU CẦU ĐẦU RA:
Trả về duy nhất một mảng JSON các đối tượng theo cấu trúc sau, không giải thích thêm:
[
  {{
    "raw_id": <id từ đầu vào>,
    "recipe_id": "<recipe_id từ đầu vào>",
    "standard_id": <ID của nguyên liệu chuẩn>,
    "raw_name": "<tên nguyên liệu thô từ đầu vào>",
    "standard_name": "<Tên nguyên liệu chuẩn>",
    "confidence": <độ tự tin từ 0.0-1.0>
  }},
  ...
]
"""

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)


def load_json_recipe_counts(json_path):
    """
    Đọc dish_ingredient.json
    Trả về dict: { recipe_id → số lượng ingredient trong JSON }
    """
    if not os.path.exists(json_path):
        print(f"⚠️  Không tìm thấy {json_path}, sẽ xử lý toàn bộ.")
        return {}

    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    counts = defaultdict(int)
    for row in data:
        counts[str(row["recipe_id"])] += 1

    print(f"✅ Đã load {len(counts)} recipe từ {json_path}")
    return dict(counts)


def generate_prompts():
    # Bước 1: Load số lượng ingredient theo recipe từ JSON
    json_recipe_counts = load_json_recipe_counts(DISH_INGREDIENT_JSON)

    # Bước 2: Kết nối DB, lấy toàn bộ ingredients
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("SELECT id, recipe_id, name FROM ingredients")
    all_rows = cursor.fetchall()
    conn.close()

    print(f"Tổng số dòng trong DB: {len(all_rows)}")

    # Bước 3: Group ingredients theo recipe_id từ DB
    db_recipe_groups = defaultdict(list)
    for row in all_rows:
        recipe_id = str(row[1])
        db_recipe_groups[recipe_id].append(row)

    # Bước 4: So sánh từng recipe
    filtered_rows = []   # các ingredients cần đưa vào prompt
    skipped_recipes = 0
    processed_recipes = 0

    for recipe_id, rows_in_db in db_recipe_groups.items():
        db_count   = len(rows_in_db)
        json_count = json_recipe_counts.get(recipe_id, 0)
        diff       = db_count - json_count

        if diff <= 2:
            # Chênh lệch 0, 1, 2 → chấp nhận, bỏ qua recipe này
            skipped_recipes += 1
        else:
            # Chênh lệch >= 3 → đưa toàn bộ ingredient của recipe này vào prompt
            filtered_rows.extend(rows_in_db)
            processed_recipes += 1

    print(f"⏭️  Bỏ qua (chênh lệch ≤ 2): {skipped_recipes} recipe")
    print(f"🆕 Cần xử lý (chênh lệch ≥ 3): {processed_recipes} recipe "
          f"({len(filtered_rows)} ingredients)")

    if not filtered_rows:
        print("🎉 Không có gì mới để xử lý!")
        return

    # Bước 5: Chia batch và tạo prompt
    batch_count = 0
    for i in range(0, len(filtered_rows), BATCH_SIZE):
        batch = filtered_rows[i:i + BATCH_SIZE]

        input_list = [
            {"raw_id": row[0], "recipe_id": row[1], "name": row[2]}
            for row in batch
        ]

        final_prompt = prompt_template.format(
            input_data=json.dumps(input_list, ensure_ascii=False, indent=2)
        )

        file_path = os.path.join(OUTPUT_DIR, f"prompt_batch_{batch_count}.txt")
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(final_prompt)

        batch_count += 1

    print(f"✅ Đã tạo {batch_count} file prompt tại thư mục: '{OUTPUT_DIR}'")


generate_prompts()

✅ Đã load 8259 recipe từ ./dish_ingredient.json
Tổng số dòng trong DB: 48333
⏭️  Bỏ qua (chênh lệch ≤ 2): 5936 recipe
🆕 Cần xử lý (chênh lệch ≥ 3): 2324 recipe (20274 ingredients)
✅ Đã tạo 676 file prompt tại thư mục: 'ai_prompts'


In [6]:
import os
import json
import re

PROMPT_DIR = 'ai_prompts'
OUTPUT_DIR = 'ai_results'

"""
Script này đọc các file prompt_batch_X.txt đã được tạo,
rồi tạo ra các file JSON template tương ứng (result_batch_X.json)
để bạn paste kết quả AI vào.

Cấu trúc mỗi file JSON:
[
  {
    "raw_id": <id>,
    "recipe_id": "<recipe_id>",
    "standard_id": null,         ← bạn điền vào
    "raw_name": "<tên gốc>",
    "standard_name": null,       ← bạn điền vào
    "confidence": null           ← bạn điền vào
  },
  ...
]
"""

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)


def extract_input_data_from_prompt(prompt_text):
    """
    Trích xuất phần JSON input_data từ nội dung file prompt.
    Tìm mảng JSON nằm sau 'DỮ LIỆU ĐẦU VÀO (JSON):'
    """
    # Tìm mảng JSON [...] trong prompt
    match = re.search(r'(\[\s*\{.*?\}\s*\])', prompt_text, re.DOTALL)
    if not match:
        return None
    try:
        data = json.loads(match.group(1))
        return data
    except json.JSONDecodeError as e:
        print(f"  ⚠️  Lỗi parse JSON: {e}")
        return None


def create_json_templates():
    if not os.path.exists(PROMPT_DIR):
        print(f"❌ Thư mục '{PROMPT_DIR}' không tồn tại. Hãy chạy generate_prompts.py trước.")
        return

    prompt_files = sorted([
        f for f in os.listdir(PROMPT_DIR)
        if f.startswith('prompt_batch_') and f.endswith('.txt')
    ], key=lambda x: int(re.search(r'\d+', x).group()))

    if not prompt_files:
        print(f"❌ Không tìm thấy file prompt nào trong '{PROMPT_DIR}'.")
        return

    print(f"🔍 Tìm thấy {len(prompt_files)} file prompt. Đang tạo JSON template...\n")

    created = 0
    skipped = 0

    for filename in prompt_files:
        # Lấy số batch từ tên file: prompt_batch_3.txt → 3
        batch_num = re.search(r'prompt_batch_(\d+)\.txt', filename).group(1)
        output_filename = f"result_batch_{batch_num}.json"
        output_path = os.path.join(OUTPUT_DIR, output_filename)

        # Nếu file đã tồn tại và có dữ liệu → bỏ qua
        if os.path.exists(output_path):
            try:
                with open(output_path, 'r', encoding='utf-8') as f:
                    existing = json.load(f)
                # Kiểm tra xem đã có dữ liệu thật chưa (standard_id != null)
                filled = any(row.get('standard_id') is not None for row in existing)
                if filled:
                    print(f"  ⏭️  {output_filename} đã có dữ liệu → bỏ qua")
                    skipped += 1
                    continue
            except Exception:
                pass  # File lỗi → tạo lại

        # Đọc file prompt
        prompt_path = os.path.join(PROMPT_DIR, filename)
        with open(prompt_path, 'r', encoding='utf-8') as f:
            prompt_text = f.read()

        # Trích xuất input data
        input_data = extract_input_data_from_prompt(prompt_text)
        if not input_data:
            print(f"  ❌ {filename}: Không thể trích xuất dữ liệu đầu vào")
            continue

        # Ghi file JSON rỗng []
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump([], f, ensure_ascii=False, indent=2)

        print(f"  ✅ {output_filename}")
        created += 1

    print(f"\n{'='*50}")
    print(f"✅ Đã tạo {created} file JSON template tại: '{OUTPUT_DIR}/'")
    if skipped:
        print(f"⏭️  Bỏ qua {skipped} file đã có dữ liệu")
    print(f"\n📋 HƯỚNG DẪN:")
    print(f"  1. Mở file prompt_batch_X.txt → copy prompt → paste vào AI")
    print(f"  2. AI trả về mảng JSON → copy kết quả")
    print(f"  3. Paste kết quả vào result_batch_X.json (ghi đè toàn bộ nội dung)")
    print(f"  4. Chạy import_results.py để đưa vào DB")


if __name__ == '__main__':
    create_json_templates()

🔍 Tìm thấy 676 file prompt. Đang tạo JSON template...

  ✅ result_batch_0.json
  ✅ result_batch_1.json
  ✅ result_batch_2.json
  ✅ result_batch_3.json
  ✅ result_batch_4.json
  ✅ result_batch_5.json
  ✅ result_batch_6.json
  ✅ result_batch_7.json
  ✅ result_batch_8.json
  ✅ result_batch_9.json
  ✅ result_batch_10.json
  ✅ result_batch_11.json
  ✅ result_batch_12.json
  ✅ result_batch_13.json
  ✅ result_batch_14.json
  ✅ result_batch_15.json
  ✅ result_batch_16.json
  ✅ result_batch_17.json
  ✅ result_batch_18.json
  ✅ result_batch_19.json
  ✅ result_batch_20.json
  ✅ result_batch_21.json
  ✅ result_batch_22.json
  ✅ result_batch_23.json
  ✅ result_batch_24.json
  ✅ result_batch_25.json
  ✅ result_batch_26.json
  ✅ result_batch_27.json
  ✅ result_batch_28.json
  ✅ result_batch_29.json
  ✅ result_batch_30.json
  ✅ result_batch_31.json
  ✅ result_batch_32.json
  ✅ result_batch_33.json
  ✅ result_batch_34.json
  ✅ result_batch_35.json
  ✅ result_batch_36.json
  ✅ result_batch_37.json
  ✅ r

In [7]:
import os
import json

OUTPUT_DIR = 'ai_results'

def get_target_indices():
    indices = set()

    # Điều kiện 1: x mod 2 == 0, 6x <= 95, m in {0,1,2,3,4,5}
    for x in range(0, 1000):
        if x % 2 == 0 and 6*x <= 95:
            for m in [0, 1, 2, 3, 4, 5]:
                indices.add(6*x + m)

    # Điều kiện 2: x mod 2 == 0, 6x > 95, 6x < 200, m in {1,2,3,4,5,6}
    for x in range(0, 1000):
        if x % 2 == 0 and 6*x > 95 and 6*x < 200:
            for m in [1, 2, 3, 4, 5, 6]:
                indices.add(6*x + m)

    return sorted(indices)


def clear_files():
    target_indices = get_target_indices()
    print(f"📋 Tổng số index cần xóa: {len(target_indices)}")
    print(f"   Danh sách: {target_indices}\n")

    if not os.path.exists(OUTPUT_DIR):
        print(f"❌ Thư mục '{OUTPUT_DIR}' không tồn tại.")
        return

    cleared = 0
    not_found = 0

    for idx in target_indices:
        filename = f"result_batch_{idx}.json"
        filepath = os.path.join(OUTPUT_DIR, filename)

        if os.path.exists(filepath):
            with open(filepath, 'w', encoding='utf-8') as f:
                json.dump([], f, ensure_ascii=False, indent=2)
            print(f"  ✅ Đã xóa nội dung: {filename}")
            cleared += 1
        else:
            print(f"  ⚠️  Không tìm thấy: {filename}")
            not_found += 1

    print(f"\n{'='*50}")
    print(f"✅ Đã xóa nội dung: {cleared} file")
    print(f"⚠️  Không tìm thấy:  {not_found} file")


if __name__ == '__main__':
    clear_files()

📋 Tổng số index cần xóa: 102
   Danh sách: [0, 1, 2, 3, 4, 5, 12, 13, 14, 15, 16, 17, 24, 25, 26, 27, 28, 29, 36, 37, 38, 39, 40, 41, 48, 49, 50, 51, 52, 53, 60, 61, 62, 63, 64, 65, 72, 73, 74, 75, 76, 77, 84, 85, 86, 87, 88, 89, 97, 98, 99, 100, 101, 102, 109, 110, 111, 112, 113, 114, 121, 122, 123, 124, 125, 126, 133, 134, 135, 136, 137, 138, 145, 146, 147, 148, 149, 150, 157, 158, 159, 160, 161, 162, 169, 170, 171, 172, 173, 174, 181, 182, 183, 184, 185, 186, 193, 194, 195, 196, 197, 198]

  ✅ Đã xóa nội dung: result_batch_0.json
  ✅ Đã xóa nội dung: result_batch_1.json
  ✅ Đã xóa nội dung: result_batch_2.json
  ✅ Đã xóa nội dung: result_batch_3.json
  ✅ Đã xóa nội dung: result_batch_4.json
  ✅ Đã xóa nội dung: result_batch_5.json
  ✅ Đã xóa nội dung: result_batch_12.json
  ✅ Đã xóa nội dung: result_batch_13.json
  ✅ Đã xóa nội dung: result_batch_14.json
  ✅ Đã xóa nội dung: result_batch_15.json
  ✅ Đã xóa nội dung: result_batch_16.json
  ✅ Đã xóa nội dung: result_batch_17.json
  ✅ 

In [1]:
import os
import json
import re
import unicodedata
from collections import defaultdict

# ============================================================
# CẤU HÌNH: mapping tên → (standard_id, standard_name)
# Match theo 2 lớp: có dấu (chính xác) + không dấu (fallback)
# ============================================================
MAPPING_RULES = [
    # (list_of_keywords, standard_id, standard_name)
    # Thứ tự quan trọng: rule cụ thể hơn đặt TRƯỚC
    (["dầu bơ", "avocado oil"],                              11830, "dầu bơ"),
    (["dầu chiên", "dầu ăn", "cooking oil", "vegetable oil",
      "oils", " oil", "dau an", "dau chien"],                11827, "dầu ăn"),
    (["bột hành", "bot hanh"],                               11832, "bột hành"),
    (["nước tương", "soy sauce", "nuoc tuong"],              11828, "nước tương"),
    (["bột canh", "bot canh"],                               11831, "bột canh"),
    (["nước hàng", "nước màu", "caramel sauce",
      "dark soy", "nuoc hang", "nuoc mau"],                  11833, "nước màu"),
    (["quế", "cinnamon", "que"],                             11829, "quế"),
]

INPUT_DIR  = "ai_results"
OUTPUT_DIR = "ai_results_fixed"


def remove_accents(text: str) -> str:
    """Bỏ dấu tiếng Việt: 'dầu ăn' → 'dau an'"""
    nfkd = unicodedata.normalize("NFD", text)
    no_accent = "".join(c for c in nfkd if not unicodedata.combining(c))
    no_accent = no_accent.replace("đ", "d").replace("Đ", "D")
    return no_accent.lower().strip()


def get_replacement(raw_name: str):
    """
    Match theo 2 lớp:
    1. Có dấu (lowercase) — chính xác
    2. Không dấu (normalize) — bắt được gõ sai dấu như 'dâu ăn', 'dau an'
    """
    name_lower     = raw_name.lower().strip()
    name_no_accent = remove_accents(raw_name)

    for keywords, std_id, std_name in MAPPING_RULES:
        for kw in keywords:
            kw_no_accent = remove_accents(kw)
            if kw in name_lower:
                return std_id, std_name
            if kw_no_accent in name_no_accent:
                return std_id, std_name
    return None


def process_file(filepath: str) -> dict:
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        return {"skipped": True}

    replaced    = 0
    removed_dup = 0
    new_data    = []
    seen: set   = set()

    for row in data:
        recipe_id = str(row.get("recipe_id", ""))
        std_id    = row.get("standard_id")
        raw_name  = row.get("raw_name", "")

        # 1. Apply mapping
        replacement = get_replacement(raw_name)
        if replacement:
            new_std_id, new_std_name = replacement
            if row.get("standard_id") != new_std_id or row.get("standard_name") != new_std_name:
                row["standard_id"]   = new_std_id
                row["standard_name"] = new_std_name
                replaced += 1
            std_id = new_std_id

        # 2. Xóa duplicate: cùng recipe_id + cùng standard_id
        dup_key = (recipe_id, std_id)
        if dup_key in seen:
            removed_dup += 1
            continue
        seen.add(dup_key)
        new_data.append(row)

    return {
        "original_count": len(data),
        "final_count":    len(new_data),
        "replaced":       replaced,
        "removed_dup":    removed_dup,
        "data":           new_data,
    }


def main():
    if not os.path.exists(INPUT_DIR):
        print(f"❌ Không tìm thấy thư mục '{INPUT_DIR}'")
        return

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    json_files = sorted(
        [f for f in os.listdir(INPUT_DIR) if f.endswith(".json")],
        key=lambda x: int(re.search(r"\d+", x).group()) if re.search(r"\d+", x) else 0,
    )

    if not json_files:
        print(f"❌ Không có file JSON trong '{INPUT_DIR}'")
        return

    print(f"🔍 Tìm thấy {len(json_files)} file JSON\n")

    total_replaced    = 0
    total_removed_dup = 0
    total_original    = 0
    total_final       = 0

    for filename in json_files:
        in_path  = os.path.join(INPUT_DIR, filename)
        out_path = os.path.join(OUTPUT_DIR, filename)

        result = process_file(in_path)

        if result.get("skipped"):
            print(f"  ⚠️  {filename}: không phải mảng JSON → bỏ qua")
            continue

        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(result["data"], f, ensure_ascii=False, indent=2)

        total_replaced    += result["replaced"]
        total_removed_dup += result["removed_dup"]
        total_original    += result["original_count"]
        total_final       += result["final_count"]

        print(
            f"  ✅ {filename}: "
            f"{result['original_count']} → {result['final_count']} bản ghi | "
            f"replaced={result['replaced']} | removed_dup={result['removed_dup']}"
        )

    print(f"\n{'='*60}")
    print(f"📊 TỔNG KẾT")
    print(f"  Tổng bản ghi ban đầu  : {total_original:,}")
    print(f"  Tổng bản ghi sau xử lý: {total_final:,}")
    print(f"  Thay thế ingredient   : {total_replaced:,}")
    print(f"  Xóa duplicate         : {total_removed_dup:,}")
    print(f"\n📁 Kết quả lưu tại: '{OUTPUT_DIR}/'")


if __name__ == "__main__":
    main()


🔍 Tìm thấy 676 file JSON

  ✅ result_batch_0.json: 30 → 28 bản ghi | replaced=1 | removed_dup=2
  ✅ result_batch_1.json: 30 → 29 bản ghi | replaced=3 | removed_dup=1
  ✅ result_batch_2.json: 30 → 28 bản ghi | replaced=3 | removed_dup=2
  ✅ result_batch_3.json: 30 → 30 bản ghi | replaced=1 | removed_dup=0
  ✅ result_batch_4.json: 30 → 29 bản ghi | replaced=1 | removed_dup=1
  ✅ result_batch_5.json: 30 → 29 bản ghi | replaced=3 | removed_dup=1
  ✅ result_batch_6.json: 30 → 28 bản ghi | replaced=2 | removed_dup=2
  ✅ result_batch_7.json: 30 → 28 bản ghi | replaced=2 | removed_dup=2
  ✅ result_batch_8.json: 30 → 30 bản ghi | replaced=3 | removed_dup=0
  ✅ result_batch_9.json: 30 → 30 bản ghi | replaced=1 | removed_dup=0
  ✅ result_batch_10.json: 30 → 28 bản ghi | replaced=6 | removed_dup=2
  ✅ result_batch_11.json: 30 → 30 bản ghi | replaced=3 | removed_dup=0
  ✅ result_batch_12.json: 30 → 29 bản ghi | replaced=1 | removed_dup=1
  ✅ result_batch_13.json: 30 → 30 bản ghi | replaced=3 | remo

In [2]:
import sqlite3
import json
from collections import defaultdict

DB_PATH = './cookpad.db'
DISH_INGREDIENT_JSON = './dish_ingredient.json'

# ── Load JSON ──────────────────────────────────────────────
with open(DISH_INGREDIENT_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)

json_recipe_counts = defaultdict(int)
for row in data:
    json_recipe_counts[str(row["recipe_id"])] += 1

print(f"✅ Recipes trong JSON: {len(json_recipe_counts)}")

# ── Load DB ────────────────────────────────────────────────
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()
cursor.execute("SELECT id, recipe_id, name FROM ingredients")
all_rows = cursor.fetchall()
conn.close()

db_recipe_groups = defaultdict(list)
for row in all_rows:
    db_recipe_groups[str(row[1])].append({
        "raw_id":    row[0],
        "recipe_id": row[1],
        "name":      row[2]
    })

print(f"✅ Recipes trong DB : {len(db_recipe_groups)}")

# ── Phân loại theo chênh lệch ──────────────────────────────
diff0 = []  # chênh lệch = 0
diff1 = []  # chênh lệch = 1
diff2 = []  # chênh lệch = 2

for recipe_id, ingredients in db_recipe_groups.items():
    db_count   = len(ingredients)
    json_count = json_recipe_counts.get(recipe_id, 0)
    diff       = db_count - json_count

    record = {
        "recipe_id":   recipe_id,
        "db_count":    db_count,
        "json_count":  json_count,
        "diff":        diff,
        "ingredients": ingredients
    }

    if diff == 0:
        diff0.append(record)
    elif diff == 1:
        diff1.append(record)
    elif diff == 2:
        diff2.append(record)

# ── In kết quả ─────────────────────────────────────────────
print(f"\n📊 Kết quả phân loại:")
print(f"  Chênh lệch = 0 : {len(diff0):>5} recipes")
print(f"  Chênh lệch = 1 : {len(diff1):>5} recipes")
print(f"  Chênh lệch = 2 : {len(diff2):>5} recipes")

# ── Lưu ra 3 file JSON ─────────────────────────────────────
for filename, dataset, label in [
    ("diff_0.json", diff0, "0"),
    ("diff_1.json", diff1, "1"),
    ("diff_2.json", diff2, "2"),
]:
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(dataset, f, ensure_ascii=False, indent=2)
    print(f"  💾 {filename} — {len(dataset)} recipes")

✅ Recipes trong JSON: 8259
✅ Recipes trong DB : 8260

📊 Kết quả phân loại:
  Chênh lệch = 0 :  1816 recipes
  Chênh lệch = 1 :  2455 recipes
  Chênh lệch = 2 :  1665 recipes
  💾 diff_0.json — 1816 recipes
  💾 diff_1.json — 2455 recipes
  💾 diff_2.json — 1665 recipes
